# Module 3: Python for Data Engineer

**EJ** · Stage 3 · Nov 19–24, 2026 · image: python-training:3.0

## Section 3.1: Datetime

### B-3.1.1

Create 30 timestamp strings in three formats, parse each group with an explicit format, and count the NaTs produced by a deliberately bad value (Lab 3.1 step 1).

In [1]:
# Will use the existing timestamps.csv file

import pandas as pd

df = pd.read_csv("../data/module3/timestamps.csv")

print(df.head())
print("Total rows:", len(df))

   event_id source_system               ts_raw
0         1           crm     2026-03-01 08:00
1         2           erp           02/03/2026
2         3           web  Mar 3, 2026 8:00 AM
3         4           crm     2026-03-03 11:00
4         5           erp           04/03/2026
Total rows: 30


The file has three columns: **event_id, source_system, and ts_raw** and 30 rows.

There are three source system groups: **crm, erp, and web**

In [2]:
# The explicit formats for each group

formats = {
    "crm": "%Y-%m-%d %H:%M",
    "erp": "%d/%m/%Y",
    "web": "%b %d, %Y %I:%M %p",
}

In [3]:
df["ts_parsed"] = pd.NaT # Start with an empty timestamp for every row

# Stop if a source_system has no format, so its rows aren't miscounted as bad
assert df["source_system"].isin(formats).all(), "Unknown source_system found"

# Parsing each group with the format
for source_system, ts_format in formats.items():
    rows = df["source_system"] == source_system

    df.loc[rows, "ts_parsed"] = pd.to_datetime(df.loc[rows, "ts_raw"], format=ts_format, errors="coerce")

In [4]:
df

,event_id,source_system,ts_raw,ts_parsed
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00
1,2,erp,02/03/2026,2026-03-02 00:00:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00
4,5,erp,04/03/2026,2026-03-04 00:00:00
5,6,web,"Mar 5, 2026 11:00 AM",2026-03-05 11:00:00
6,7,crm,2026-03-05 14:00,2026-03-05 14:00:00
7,8,erp,31/02/2026,NaT
8,9,web,"Mar 7, 2026 2:00 PM",2026-03-07 14:00:00
9,10,crm,2026-03-07 17:00,2026-03-07 17:00:00


In [5]:
bad = df[df["ts_parsed"].isna()] # returns only the rows where ts_parsed is NaT
bad

,event_id,source_system,ts_raw,ts_parsed
7,8,erp,31/02/2026,NaT
20,21,web,"Marz 9, 2026 3:00 PM",NaT


Among the 30 rows, only two are not valid timestamps:

- event_id 8 - **31/02/2026**: there is no february 31
- event_id 21 - **Marz 9, 2026** 3:00 PM: Marz is not a valid month name

In [6]:
assert len(bad) == 2, f"Expected 2 bad timestamps, got {len(bad)}"

### B-3.1.2

Make the datetimes timezone-aware, convert to UTC, and show both side by side (Lab 3.1 step 2).

In [7]:
# Assigning a local timezone
df["ts_local"] = df["ts_parsed"].dt.tz_localize("Asia/Manila")
df.head()

,event_id,source_system,ts_raw,ts_parsed,ts_local
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00,2026-03-01 08:00:00+08:00
1,2,erp,02/03/2026,2026-03-02 00:00:00,2026-03-02 00:00:00+08:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00,2026-03-03 08:00:00+08:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00,2026-03-03 11:00:00+08:00
4,5,erp,04/03/2026,2026-03-04 00:00:00,2026-03-04 00:00:00+08:00


In [8]:
# Converting the local UTC+8 to UTC+0
df["ts_utc"] = df["ts_local"].dt.tz_convert("UTC")
df.head()

,event_id,source_system,ts_raw,ts_parsed,ts_local,ts_utc
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00,2026-03-01 08:00:00+08:00,2026-03-01 00:00:00+00:00
1,2,erp,02/03/2026,2026-03-02 00:00:00,2026-03-02 00:00:00+08:00,2026-03-01 16:00:00+00:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00,2026-03-03 08:00:00+08:00,2026-03-03 00:00:00+00:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00,2026-03-03 11:00:00+08:00,2026-03-03 03:00:00+00:00
4,5,erp,04/03/2026,2026-03-04 00:00:00,2026-03-04 00:00:00+08:00,2026-03-03 16:00:00+00:00


In [9]:
# The two timezone's side-by-side

df[["event_id", "ts_local", "ts_utc"]].head()

,event_id,ts_local,ts_utc
0,1,2026-03-01 08:00:00+08:00,2026-03-01 00:00:00+00:00
1,2,2026-03-02 00:00:00+08:00,2026-03-01 16:00:00+00:00
2,3,2026-03-03 08:00:00+08:00,2026-03-03 00:00:00+00:00
3,4,2026-03-03 11:00:00+08:00,2026-03-03 03:00:00+00:00
4,5,2026-03-04 00:00:00+08:00,2026-03-03 16:00:00+00:00


For event_id 2, **March 2 at 00:00** in Manila or 12 AM becomes **March 1 at 16:00** or 4PM in UTC.

Because Manila is **UTC+8**, any local time before 8:00 AM falls on the previous day in UTC

### B-3.1.3

Derive year, month, weekday, and hour columns, and resample to daily counts (Lab 3.1 step 3).

In [10]:
# Using the same ts_local so it follows local time

df["year"] = df["ts_local"].dt.year
df["month"] = df["ts_local"].dt.month
df["weekday"] = df["ts_local"].dt.day_name()
df["hour"] = df["ts_local"].dt.hour

df[["event_id", "ts_local", "year", "month", "weekday", "hour"]].head() 

,event_id,ts_local,year,month,weekday,hour
0,1,2026-03-01 08:00:00+08:00,2026.0,3.0,Sunday,8.0
1,2,2026-03-02 00:00:00+08:00,2026.0,3.0,Monday,0.0
2,3,2026-03-03 08:00:00+08:00,2026.0,3.0,Tuesday,8.0
3,4,2026-03-03 11:00:00+08:00,2026.0,3.0,Tuesday,11.0
4,5,2026-03-04 00:00:00+08:00,2026.0,3.0,Wednesday,0.0


Since there are two NaT values, the output shows float versions for the year, month, and hour. Missing values are stored as NaN, which is a float, so pandas turns the whole column into float to fit them.

In [11]:
# Resampling to daily counts

daily_counts = (df.resample("D", on="ts_local").size().rename("event_count"))
# D groups events into calendar days using ts_local

print("Days:", len(daily_counts))
daily_counts

Days: 22


ts_local
2026-03-01 00:00:00+08:00    1
2026-03-02 00:00:00+08:00    1
2026-03-03 00:00:00+08:00    2
2026-03-04 00:00:00+08:00    1
2026-03-05 00:00:00+08:00    2
2026-03-06 00:00:00+08:00    0
2026-03-07 00:00:00+08:00    2
2026-03-08 00:00:00+08:00    1
2026-03-09 00:00:00+08:00    2
2026-03-10 00:00:00+08:00    1
2026-03-11 00:00:00+08:00    2
2026-03-12 00:00:00+08:00    1
2026-03-13 00:00:00+08:00    1
2026-03-14 00:00:00+08:00    1
2026-03-15 00:00:00+08:00    1
2026-03-16 00:00:00+08:00    1
2026-03-17 00:00:00+08:00    1
2026-03-18 00:00:00+08:00    2
2026-03-19 00:00:00+08:00    1
2026-03-20 00:00:00+08:00    2
2026-03-21 00:00:00+08:00    1
2026-03-22 00:00:00+08:00    1
Name: event_count, dtype: int64

There are **28 valid events over 22 days** (the 2 invalid timestamps are excluded). Mar 6 shows 0 because resample lists every calendar day in the range, even days with no events.

In [12]:
valid_count = df["ts_local"].notna().sum()
assert daily_counts.sum() == valid_count, "Daily counts don't add up to the valid events"

### B-3.1.4

Compute the time between two events with `timedelta` (Lab 3.1 step 4).

In [13]:
# Will use the evets.csv file for this, specifically the ordered_at and shipped_at columns

events = pd.read_csv("../data/module3/events.csv")
iso_format = "%Y-%m-%dT%H:%M:%S" # original format of the timestamp

# Parse the format into the ordered_at and shipped_at events
for column in ["ordered_at", "shipped_at"]:
    events[column] = pd.to_datetime(events[column], format=iso_format, errors="coerce")

print(events.shape)
events.head()

(10, 3)


,event_id,ordered_at,shipped_at
0,E001,2026-03-02 15:45:00,2026-03-03 18:45:00
1,E002,2026-03-03 10:45:00,2026-03-06 09:45:00
2,E003,2026-03-04 16:15:00,2026-03-06 09:15:00
3,E004,2026-03-05 13:45:00,2026-03-10 01:45:00
4,E005,2026-03-06 15:45:00,NaT


The file has **three columns and 10 rows**. One row has a **missing** shipped_at value.

In [14]:
# timedelta handles a missing value (event 5)
events["order_to_ship_duration"] = events["shipped_at"] - events["ordered_at"]

events

,event_id,ordered_at,shipped_at,order_to_ship_duration
0,E001,2026-03-02 15:45:00,2026-03-03 18:45:00,1 days 03:00:00
1,E002,2026-03-03 10:45:00,2026-03-06 09:45:00,2 days 23:00:00
2,E003,2026-03-04 16:15:00,2026-03-06 09:15:00,1 days 17:00:00
3,E004,2026-03-05 13:45:00,2026-03-10 01:45:00,4 days 12:00:00
4,E005,2026-03-06 15:45:00,NaT,NaT
5,E006,2026-03-07 15:15:00,2026-03-09 04:15:00,1 days 13:00:00
6,E007,2026-03-08 08:00:00,2026-03-11 04:00:00,2 days 20:00:00
7,E008,2026-03-09 10:15:00,2026-03-09 19:15:00,0 days 09:00:00
8,E009,2026-03-10 17:15:00,2026-03-11 00:15:00,0 days 07:00:00
9,E010,2026-03-11 17:00:00,2026-03-15 14:00:00,3 days 21:00:00


In [15]:
print("Average:", events["order_to_ship_duration"].mean())
print("Longest:", events["order_to_ship_duration"].max())

Average: 2 days 03:13:20
Longest: 4 days 12:00:00


E005 has no **shipped_at**, so it parses to NaT. Any subtraction with NaT gives NaT, so its order_to_ship_duration is NaT too. 

**.mean() and .max() skip NaT**, which means the average covers 9 orders, not 10.

**Why silent parsing failures are dangerous and how to make them loud?**

Silent parsing are dangerious because a report can look complete even when it leaves out records. In `timestamps.csv`, `errors="coerce"` turned two invalid timestamps into `NaT` and the script kept running, but he daily counts included only 28 out of 30 events.


I could've just used `errors="raise"` to stop when pandas finds an invalid timestamp and make the failures loud. Instead, I used `errors="coerce"` and made it loud by counting **NaTs**, printing the failed rows, and asserting the count matched the bad inputs I'd found.

## Section 3.2: JSON, CSV, and Excel

### 3.2 Design choices

**Data:** the provided `orders_nested.json` (12 orders, nested customer and items) and `large_events.csv` (50,000 rows), from the module's data folder.

**Package:** openpyxl 3.1.5, added to requirements.txt for the Excel round trip.

**One row = one item** from an order's `items` list. Each row also carries the order's `order_id` and `ordered_at`, plus the customer's `id` and `name`. Email and city are left out because this lab doesn't need them.

**Prediction:** 36 rows, the total number of items across the 12 orders (counted in code below).

**Chunked read**: large_events.csv (provided, 50,000 rows) is read in chunks of 10,000 rows, so the loop should run 5 times. In a real pipeline, chunking lets you process files too big to fit in memory, because only one chunk is loaded at a time. 10,000 keeps each chunk small while needing only a few loops.

### B-3.2.1

Create a small nested JSON file, flatten it to one row per item, and save it to CSV and Excel (Lab 3.2 step 1).

In [19]:
# Will use the existing orders_nested.json file

import json
import pandas as pd

with open("../data/module3/orders_nested.json", encoding="utf-8") as file:
    orders = json.load(file) # reads the file into a python list

In [20]:
# Since each item should be its own row, there should be 36 rows in total (prediction)

expected_rows = sum(len(order["items"]) for order in orders)
expected_rows

36

In [21]:
print("Orders:", len(orders))
print(json.dumps(orders[0], indent=2)) # one order sample

Orders: 12
{
  "order_id": "A-1001",
  "ordered_at": "2026-04-01",
  "customer": {
    "id": "C-004",
    "name": "Customer A",
    "email": "customera@example.test",
    "city": "Davao City"
  },
  "items": [
    {
      "sku": "P-102",
      "name": "Mouse",
      "qty": 1,
      "unit_price": 14.0
    },
    {
      "sku": "P-105",
      "name": "USB Cable",
      "qty": 1,
      "unit_price": 6.5
    },
    {
      "sku": "P-104",
      "name": "Desk Lamp",
      "qty": 1,
      "unit_price": 18.0
    },
    {
      "sku": "P-103",
      "name": "Keyboard",
      "qty": 2,
      "unit_price": 25.0
    }
  ]
}


In [22]:
# Flattening the items list, one row per item

items = pd.json_normalize(orders, record_path="items",
    meta=["order_id", "ordered_at", ["customer", "id"], ["customer", "name"], ["customer", "email"], ["customer", "city"]],
    sep="_",
)

items.head()

,sku,name,qty,unit_price,order_id,ordered_at,customer_id,customer_name,customer_email,customer_city
0,P-102,Mouse,1,14.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
1,P-105,USB Cable,1,6.5,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
2,P-104,Desk Lamp,1,18.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
3,P-103,Keyboard,2,25.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
4,P-103,Keyboard,4,25.0,A-1002,2026-04-02,C-001,Customer B,customerb@example.test,Baguio


In [23]:
# Checking the quality of the flattened rows

quality = []
for col in items.columns:
    quality.append({
        "column": col,
        "dtype": str(items[col].dtype),
        "nulls": int(items[col].isna().sum()),
        "unique": items[col].nunique(),
    })

pd.DataFrame(quality)

,column,dtype,nulls,unique
0,sku,object,0,6
1,name,object,0,6
2,qty,int64,0,5
3,unit_price,float64,0,6
4,order_id,object,0,12
5,ordered_at,object,0,12
6,customer_id,object,0,8
7,customer_name,object,0,12
8,customer_email,object,7,10
9,customer_city,object,0,4


There are 7 rows which have a null `customer_email`. The code below checks the original json file which specific orders are these. 

In [24]:
[order for order in orders if order["customer"]["email"] is None]

[{'order_id': 'A-1004',
  'ordered_at': '2026-04-04',
  'customer': {'id': 'C-001',
   'name': 'Customer D',
   'email': None,
   'city': 'Davao City'},
  'items': [{'sku': 'P-103', 'name': 'Keyboard', 'qty': 1, 'unit_price': 25.0},
   {'sku': 'P-104', 'name': 'Desk Lamp', 'qty': 1, 'unit_price': 18.0},
   {'sku': 'P-105', 'name': 'USB Cable', 'qty': 2, 'unit_price': 6.5}]},
 {'order_id': 'A-1010',
  'ordered_at': '2026-04-10',
  'customer': {'id': 'C-006',
   'name': 'Customer J',
   'email': None,
   'city': 'Quezon City'},
  'items': [{'sku': 'P-100', 'name': 'Notebook', 'qty': 5, 'unit_price': 4.5},
   {'sku': 'P-104', 'name': 'Desk Lamp', 'qty': 3, 'unit_price': 18.0},
   {'sku': 'P-103', 'name': 'Keyboard', 'qty': 1, 'unit_price': 25.0},
   {'sku': 'P-101', 'name': 'Pen', 'qty': 4, 'unit_price': 1.2}]}]

Seems like `A-1004` contains 3 items and `A-1010` contains 4 items, resulting into the 7 null emails in our checking. `json_normalize` copies each order-level field onto every item row, so one missing email becomes one null per item in that order.

In [25]:
assert len(items) == expected_rows, "Number of flattened rows do not match the expected number of rows"

# Saving the result into CSV and Excel

items.to_csv("../data/module3/orders_nested_flattened.csv", index=False)
items.to_excel("../data/module3/orders_nested_flattened.xlsx", index=False)

### B-3.2.2

Read the CSV back with explicit `dtype` and `usecols`, read the Excel file back, and compare the three versions (Lab 3.2 step 2).

In [26]:
# Reading the CSV with explicit dtype and usecols

column_types = {
    "order_id": "string",
    "sku": "string",
    "qty": "int64",
    "unit_price": "float64",
    "customer_email": "string",
}

columns = list(column_types)

csv_items = pd.read_csv("../data/module3/orders_nested_flattened.csv", usecols=columns, dtype=column_types)
excel_items = pd.read_excel("../data/module3/orders_nested_flattened.xlsx", usecols=columns)

In [36]:
# Comparing column types between the three versions

pd.DataFrame({
    "Original": items[columns].dtypes,
    "CSV": csv_items[columns].dtypes,
    "Excel": excel_items[columns].dtypes,
})

,Original,CSV,Excel
order_id,object,string[python],object
sku,object,string[python],object
qty,int64,int64,int64
unit_price,float64,float64,float64
customer_email,object,string[python],object


The datatypes between the original normalized json file and the excel file remain exactly the same. For the CSV file, since I explicitly set the dtype, `order_id`, `sku`, and `customer_email` shows `string[python]` instead.

In [38]:
# Comparing if the row count and number of missing values stayed the same

tables = {"Original": items, "CSV": csv_items, "Excel": excel_items}

missing_emails = items[items["customer_email"].isna()]
first_missing_row = missing_emails.index[0]

summary = {}
for label, table in tables.items():
    summary[label] = {"Rows": len(table)}

    for column in csv_items.columns:
        summary[label][f"Missing {column}"] = table[column].isna().sum()

    email = table.loc[first_missing_row, "customer_email"]
    summary[label]["Missing email dtype"] = repr(email)

pd.DataFrame(summary)

,Original,CSV,Excel
Rows,36,36,36
Missing sku,0,0,0
Missing qty,0,0,0
Missing unit_price,0,0,0
Missing order_id,0,0,0
Missing customer_email,7,7,7
Missing email dtype,None,<NA>,nan


In [31]:
# Checking if the resuls match

original_typed = items[csv_items.columns].astype(column_types)
assert csv_items.equals(original_typed), "CSV round trip changed the data"
assert excel_items.equals(items[excel_items.columns]), "Excel round trip changed the data"
print("CSV and Excel both match the original")

CSV and Excel both match the original


**What changes when data makes a round trip between formats?**

The row and missing count for each three versions matched, including the 7 missing emails. What changed was the **CSV's text columns** and how pandas displayed a missing email. For a missing email, `json.load()` converted JSON `null` to Python `None`. Pandas read the empty CSV field as `<NA>` and the blank Excel cell as `NaN`.

### B-3.2.3

Read a larger CSV in chunks and compute a running total (Lab 3.2 step 3).

In [43]:
# Will use the provided large_events file which contains 50K events

chunks = pd.read_csv("../data/module3/large_events.csv", usecols=["amount"], chunksize=10_000)

running_total = 0 # Will hold the total of amounts

for chunk_number, chunk in enumerate(chunks, start=1):
    running_total += chunk["amount"].sum()
    print(f"After chunk {chunk_number}: {running_total:,.2f}")

After chunk 1: 2,519,323.26
After chunk 2: 5,046,021.47
After chunk 3: 7,551,027.03
After chunk 4: 10,048,405.97
After chunk 5: 12,542,076.33


`usecols=["amount"]` loaded only the column I needed, so each chunk held 1 of the 3 columns. Chunking limits how many rows are in memory, and usecols limits how many columns.

In [44]:
full_total = pd.read_csv("../data/module3/large_events.csv")["amount"].sum()

assert chunk_number == 5, f"Expected 5 chunks, got {chunk_number}"
assert round(running_total, 2) == round(full_total, 2), "Chunked total doesn't match the full read"
print(f"Chunks: {chunk_number} | Chunked total: {running_total:,.2f} | Full read: {full_total:,.2f}")

Chunks: 5 | Chunked total: 12,542,076.33 | Full read: 12,542,076.33


## Section 3.3: Avro

### B-3.3.1

Define an Avro schema for the flattened order-item table with one optional field (Lab 3.3 step 1).

### B-3.3.2

Write `orders.avro`, read it back, and load it into a DataFrame (Lab 3.3 step 2).

### B-3.3.3

Compare file sizes of CSV, JSON, and Avro for the same data in a table (Lab 3.3 step 3).

### B-3.3.4

Add a field to the schema and show reading old data with the new schema (Lab 3.3 step 4).

## Section 3.4: Web scraping (local sandbox)

### B-3.4.1

Add the `sandbox-site` service under the `scrape` profile and start it (Lab 3.4 step 1).

### B-3.4.2

Fetch and parse the site's `robots.txt`, and write down what it allows for the trainee's user agent and the date checked (Lab 3.4 step 2).

### B-3.4.3

Scrape one listing page, extracting at least 4 fields per item (Lab 3.4 step 3).

### B-3.4.4

Follow pagination across all pages with a delay and a descriptive user-agent string (Lab 3.4 step 4).

### B-3.4.5

Add a check that skips disallowed URLs and show it refusing the `/admin/` path (Lab 3.4 step 5).

### B-3.4.6

Clean the results (price to float) and save them to `data/products.csv` (Lab 3.4 step 6).

## Diagrams

## What broke and how I fixed it

## Reflection

A reflection of 3 to 5 sentences